In [1]:
from pathlib import Path

print("Python environment working.")
print("Current working directory:", Path.cwd())

Python environment working.
Current working directory: C:\Users\HP\Documents\Projects\Brain_Tumor_XAI\notebooks


In [2]:
from pathlib import Path

# Current notebook folder
NOTEBOOK_DIR = Path.cwd()

# Project root = one level above notebooks/
PROJECT_ROOT = NOTEBOOK_DIR.parent

# BRISC dataset paths
DATASET_ROOT = PROJECT_ROOT / "data" / "raw" / "brisc2025"
CLASSIFICATION_ROOT = DATASET_ROOT / "classification_task"
SEGMENTATION_ROOT = DATASET_ROOT / "segmentation_task"

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)

print("\nPath checks:")
print("Dataset exists:", DATASET_ROOT.exists())
print("Classification task exists:", CLASSIFICATION_ROOT.exists())
print("Segmentation task exists:", SEGMENTATION_ROOT.exists())

Project root: C:\Users\HP\Documents\Projects\Brain_Tumor_XAI
Dataset root: C:\Users\HP\Documents\Projects\Brain_Tumor_XAI\data\raw\brisc2025

Path checks:
Dataset exists: True
Classification task exists: True
Segmentation task exists: True


In [3]:
print("BRISC top-level contents:\n")

for item in sorted(DATASET_ROOT.iterdir()):
    item_type = "DIR " if item.is_dir() else "FILE"
    print(f"{item_type} - {item.name}")

BRISC top-level contents:

DIR  - classification_task
FILE - manifest.csv
FILE - manifest.csv.sha256
FILE - manifest.json
FILE - manifest.json.sha256
FILE - README.md
DIR  - segmentation_task


## 1. Classification Task — Class and Split Counts

In [4]:
# Count classification images in each split and class

classification_counts = {}

for split in ["train", "test"]:
    split_path = CLASSIFICATION_ROOT / split
    classification_counts[split] = {}

    print(f"\n{split.upper()} SPLIT")
    print("-" * 40)

    total = 0

    for class_dir in sorted(split_path.iterdir()):
        if class_dir.is_dir():
            files = [f for f in class_dir.iterdir() if f.is_file()]
            count = len(files)

            classification_counts[split][class_dir.name] = count
            total += count

            print(f"{class_dir.name:<15}: {count}")

    print(f"{'TOTAL':<15}: {total}")


TRAIN SPLIT
----------------------------------------
glioma         : 1147
meningioma     : 1329
no_tumor       : 1067
pituitary      : 1457
TOTAL          : 5000

TEST SPLIT
----------------------------------------
glioma         : 254
meningioma     : 306
no_tumor       : 140
pituitary      : 300
TOTAL          : 1000


## 2. Segmentation Task — Image and Mask Counts

In [5]:
# Count segmentation images and masks

for split in ["train", "test"]:
    images_path = SEGMENTATION_ROOT / split / "images"
    masks_path = SEGMENTATION_ROOT / split / "masks"

    image_files = [f for f in images_path.iterdir() if f.is_file()]
    mask_files = [f for f in masks_path.iterdir() if f.is_file()]

    print(f"\n{split.upper()} SPLIT")
    print("-" * 40)
    print("Images :", len(image_files))
    print("Masks  :", len(mask_files))
    print("Difference:", len(image_files) - len(mask_files))


TRAIN SPLIT
----------------------------------------
Images : 3933
Masks  : 3933
Difference: 0

TEST SPLIT
----------------------------------------
Images : 860
Masks  : 860
Difference: 0


## 3. File Extension Inspection

In [6]:
from collections import Counter

print("CLASSIFICATION FILE EXTENSIONS")
print("-" * 40)

for split in ["train", "test"]:
    extensions = Counter()

    for class_dir in (CLASSIFICATION_ROOT / split).iterdir():
        if class_dir.is_dir():
            for file in class_dir.iterdir():
                if file.is_file():
                    extensions[file.suffix.lower()] += 1

    print(f"{split.capitalize()}:", dict(extensions))


print("\nSEGMENTATION FILE EXTENSIONS")
print("-" * 40)

for split in ["train", "test"]:
    image_extensions = Counter(
        f.suffix.lower()
        for f in (SEGMENTATION_ROOT / split / "images").iterdir()
        if f.is_file()
    )

    mask_extensions = Counter(
        f.suffix.lower()
        for f in (SEGMENTATION_ROOT / split / "masks").iterdir()
        if f.is_file()
    )

    print(f"\n{split.upper()}")
    print("Images:", dict(image_extensions))
    print("Masks :", dict(mask_extensions))

CLASSIFICATION FILE EXTENSIONS
----------------------------------------
Train: {'.jpg': 5000}
Test: {'.jpg': 1000}

SEGMENTATION FILE EXTENSIONS
----------------------------------------

TRAIN
Images: {'.jpg': 3933}
Masks : {'.png': 3933}

TEST
Images: {'.jpg': 860}
Masks : {'.png': 860}


## 4. Segmentation Image–Mask Pairing Verification

In [7]:
def check_image_mask_pairs(split):
    images_dir = SEGMENTATION_ROOT / split / "images"
    masks_dir = SEGMENTATION_ROOT / split / "masks"

    # Compare filename stems, ignoring .jpg vs .png
    image_stems = {f.stem for f in images_dir.iterdir() if f.is_file()}
    mask_stems = {f.stem for f in masks_dir.iterdir() if f.is_file()}

    missing_masks = sorted(image_stems - mask_stems)
    missing_images = sorted(mask_stems - image_stems)

    print(f"\n{split.upper()} SPLIT")
    print("-" * 40)
    print("Image filenames :", len(image_stems))
    print("Mask filenames  :", len(mask_stems))
    print("Images without masks :", len(missing_masks))
    print("Masks without images :", len(missing_images))

    if not missing_masks and not missing_images:
        print("Pairing status: PERFECT MATCH")
    else:
        print("Pairing status: MISMATCH FOUND")

        if missing_masks:
            print("\nFirst few images without masks:")
            print(missing_masks[:10])

        if missing_images:
            print("\nFirst few masks without images:")
            print(missing_images[:10])


check_image_mask_pairs("train")
check_image_mask_pairs("test")


TRAIN SPLIT
----------------------------------------
Image filenames : 3933
Mask filenames  : 3933
Images without masks : 0
Masks without images : 0
Pairing status: PERFECT MATCH

TEST SPLIT
----------------------------------------
Image filenames : 860
Mask filenames  : 860
Images without masks : 0
Masks without images : 0
Pairing status: PERFECT MATCH


## 5. Image Readability and Corruption Check

In [8]:
from PIL import Image

def check_readability(file_paths):
    bad_files = []

    for file_path in file_paths:
        try:
            with Image.open(file_path) as img:
                img.load()   # Forces the image data to actually be decoded
        except Exception as e:
            bad_files.append((file_path, str(e)))

    return bad_files


# --------------------------------------------------
# 1. Classification files
# --------------------------------------------------

classification_files = []

for split in ["train", "test"]:
    for class_dir in (CLASSIFICATION_ROOT / split).iterdir():
        if class_dir.is_dir():
            classification_files.extend(
                f for f in class_dir.iterdir() if f.is_file()
            )


# --------------------------------------------------
# 2. Segmentation MRI images
# --------------------------------------------------

segmentation_images = []

for split in ["train", "test"]:
    images_dir = SEGMENTATION_ROOT / split / "images"
    segmentation_images.extend(
        f for f in images_dir.iterdir() if f.is_file()
    )


# --------------------------------------------------
# 3. Segmentation masks
# --------------------------------------------------

segmentation_masks = []

for split in ["train", "test"]:
    masks_dir = SEGMENTATION_ROOT / split / "masks"
    segmentation_masks.extend(
        f for f in masks_dir.iterdir() if f.is_file()
    )


print("Checking classification images...")
bad_classification = check_readability(classification_files)

print("Checking segmentation images...")
bad_segmentation_images = check_readability(segmentation_images)

print("Checking segmentation masks...")
bad_segmentation_masks = check_readability(segmentation_masks)


print("\nREADABILITY REPORT")
print("-" * 45)

print("Classification files checked :", len(classification_files))
print("Unreadable classification files:", len(bad_classification))

print("\nSegmentation images checked :", len(segmentation_images))
print("Unreadable segmentation images:", len(bad_segmentation_images))

print("\nSegmentation masks checked :", len(segmentation_masks))
print("Unreadable segmentation masks:", len(bad_segmentation_masks))

Checking classification images...
Checking segmentation images...
Checking segmentation masks...

READABILITY REPORT
---------------------------------------------
Classification files checked : 6000
Unreadable classification files: 0

Segmentation images checked : 4793
Unreadable segmentation images: 0

Segmentation masks checked : 4793
Unreadable segmentation masks: 0


## 6. Image Dimensions and Channel/Mode Inspection

In [9]:
from collections import Counter
from PIL import Image


def inspect_image_properties(file_paths):
    dimensions = Counter()
    modes = Counter()

    for file_path in file_paths:
        with Image.open(file_path) as img:
            dimensions[img.size] += 1
            modes[img.mode] += 1

    return dimensions, modes


# Classification images
class_dimensions, class_modes = inspect_image_properties(
    classification_files
)

# Segmentation MRI images
seg_dimensions, seg_modes = inspect_image_properties(
    segmentation_images
)

# Segmentation masks
mask_dimensions, mask_modes = inspect_image_properties(
    segmentation_masks
)


print("CLASSIFICATION IMAGES")
print("-" * 45)
print("Unique dimensions:", len(class_dimensions))
print("Dimensions:")
for size, count in class_dimensions.most_common():
    print(f"  {size}: {count}")

print("\nImage modes:")
for mode, count in class_modes.items():
    print(f"  {mode}: {count}")


print("\nSEGMENTATION MRI IMAGES")
print("-" * 45)
print("Unique dimensions:", len(seg_dimensions))
print("Dimensions:")
for size, count in seg_dimensions.most_common():
    print(f"  {size}: {count}")

print("\nImage modes:")
for mode, count in seg_modes.items():
    print(f"  {mode}: {count}")


print("\nSEGMENTATION MASKS")
print("-" * 45)
print("Unique dimensions:", len(mask_dimensions))
print("Dimensions:")
for size, count in mask_dimensions.most_common():
    print(f"  {size}: {count}")

print("\nMask modes:")
for mode, count in mask_modes.items():
    print(f"  {mode}: {count}")

CLASSIFICATION IMAGES
---------------------------------------------
Unique dimensions: 183
Dimensions:
  (512, 512): 4881
  (369, 369): 363
  (216, 369): 354
  (256, 256): 32
  (187, 369): 21
  (225, 225): 20
  (202, 369): 19
  (210, 369): 12
  (236, 236): 11
  (442, 442): 10
  (455, 500): 5
  (554, 554): 5
  (201, 251): 5
  (200, 223): 4
  (212, 237): 4
  (210, 240): 4
  (341, 395): 4
  (213, 237): 3
  (472, 546): 2
  (441, 427): 2
  (485, 510): 2
  (507, 605): 2
  (338, 345): 2
  (369, 398): 2
  (591, 650): 2
  (351, 398): 2
  (341, 377): 2
  (491, 624): 2
  (314, 358): 2
  (257, 307): 2
  (300, 345): 2
  (291, 340): 2
  (318, 354): 2
  (315, 341): 2
  (278, 306): 2
  (312, 401): 2
  (300, 412): 2
  (205, 251): 2
  (204, 249): 2
  (323, 342): 2
  (216, 216): 2
  (341, 372): 2
  (273, 326): 2
  (305, 337): 2
  (304, 410): 2
  (208, 243): 2
  (200, 252): 2
  (480, 480): 2
  (224, 219): 2
  (200, 235): 2
  (209, 241): 2
  (211, 239): 2
  (215, 234): 2
  (214, 216): 2
  (214, 235): 2
  (

## 7. Segmentation Image–Mask Dimension Matching

In [10]:
from PIL import Image

def check_pair_dimensions(split):
    images_dir = SEGMENTATION_ROOT / split / "images"
    masks_dir = SEGMENTATION_ROOT / split / "masks"

    mismatches = []

    image_files = sorted(
        f for f in images_dir.iterdir() if f.is_file()
    )

    for image_path in image_files:
        mask_path = masks_dir / f"{image_path.stem}.png"

        with Image.open(image_path) as img, Image.open(mask_path) as mask:
            if img.size != mask.size:
                mismatches.append({
                    "image": image_path.name,
                    "image_size": img.size,
                    "mask": mask_path.name,
                    "mask_size": mask.size
                })

    print(f"\n{split.upper()} SPLIT")
    print("-" * 45)
    print("Pairs checked:", len(image_files))
    print("Dimension mismatches:", len(mismatches))

    if len(mismatches) == 0:
        print("Dimension status: PERFECT MATCH")
    else:
        print("Dimension status: MISMATCH FOUND")
        print("\nFirst few mismatches:")
        for item in mismatches[:10]:
            print(item)

    return mismatches


train_dimension_mismatches = check_pair_dimensions("train")
test_dimension_mismatches = check_pair_dimensions("test")


TRAIN SPLIT
---------------------------------------------
Pairs checked: 3933
Dimension mismatches: 0
Dimension status: PERFECT MATCH

TEST SPLIT
---------------------------------------------
Pairs checked: 860
Dimension mismatches: 0
Dimension status: PERFECT MATCH
